In [ ]:
import os
import re
import sys
import warnings
import pandas as pd
import numpy as np
import torch

from joblib import Parallel, delayed

import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import FunctionTransformer

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import StratifiedKFold

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.word2vec import Word2Vec
from gensim.models.doc2vec import TaggedDocument, Doc2Vec
from sentence_transformers import SentenceTransformer

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier

from scipy.stats import shapiro
from scipy.stats import levene

In [2]:
# Load a pre-trained SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Move the model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
if torch.cuda.is_available():
    print('GPU : ', torch.cuda.get_device_name(0))

# Élimination des avertissements
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

GPU :  NVIDIA GeForce RTX 3070


In [17]:
punct = string.punctuation.replace('-', '')
stopw = stopwords.words('english') + list(punct)
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

def tokenize_remove_stop_words(text: str):
    return [token for token in word_tokenize(text) if 
            token.lower() not in stopw and
            len(token) > 2 and  # Mots de moins de 2 lettres
            not (bool(re.search(r'\d', token))) and # Mots contenant des chiffres
            not (any(char in punct for char in token))] # Mots contenant des signes de ponctuation

def vectorize_word2vec(corpus, w2v_model):    
    def vectorize(document_tokenized):
        words_vecs = [w2v_model.wv[word] for word in document_tokenized if word in w2v_model.wv]
        if len(words_vecs) == 0:
            return np.zeros(w2v_model.vector_size)
        words_vecs = np.array(words_vecs)
        return words_vecs.mean(axis=0)
    
    tokenized_corpus = [list(tokenize_remove_stop_words(doc)) for doc in corpus]
    X = np.array([vectorize(doc) for doc in tokenized_corpus])
    return X

In [18]:
features_tfidf = [1000] #, 2000, 3000, 4000, 5000]
features_w2v = [100, 200, 300, 400, 500]

classifiers = [
    LogisticRegression(), 
    LinearSVC(dual="auto"),
    MultinomialNB(),
    RandomForestClassifier(n_estimators=32)
]

In [ ]:
results_training = []
results_test = []

def train_and_evaluate(dataset):
    print('Entraînement pour le jeu de données : ', dataset)

    ratio_incels = dataset[-8:-6]

    ### Lecture du jeu de données et partitionnement de celles-ci en ensembles d'entraînement et de test
    train = pd.read_csv(f'../data/training_datasets/{dataset}').sample(100)
    train['category'] = train['category'].apply(lambda x: 1 if x == 'incel' else 0)

    X_train, y_train = train.text_post.astype('str'), train.category

    ### Définition des fonctions de vectorisation    
    # Charger les modèles Word2Vec
    word2vec_transformers = [FunctionTransformer(
        lambda x: vectorize_word2vec(
            x, 
            w2v_model = Word2Vec.load(
                f"../word2vec_models/w2v_{i}_dim_{ratio_incels}pc_incels.model")
        )
    ) for i in features_w2v]

    vectorizers = {
        # TF-IDF 
        'TfidfVectorizer' : TfidfVectorizer(            
            stop_words=stopw,
            tokenizer=word_tokenize,
            min_df=2,
            token_pattern=None
        ),

        # Word2Vec 
        'Word2Vec__300' : word2vec_transformers[2],

        # Sentence-BERT
        'SentenceTransformer': FunctionTransformer(
            lambda x: model.encode(
                x.squeeze().astype(str).values,
                batch_size=64,
                convert_to_numpy=True,
                show_progress_bar=True,
                device=device)
        )
    }

    tf_idf_param_grid = [
        {
            "vectorizer__max_features": features_tfidf,
            "classify" : classifiers
        }
    ]

    w2v_param_grid = [
            {
            "classify" : classifiers
        }
    ]

    sbert_param_grid = [
            {
            "classify" : classifiers
        }
    ]

    param_grid = {
        'TfidfVectorizer' : tf_idf_param_grid,
        'Word2Vec__100' : w2v_param_grid,
        'Word2Vec__200' : w2v_param_grid,
        'Word2Vec__300' : w2v_param_grid,
        'Word2Vec__400' : w2v_param_grid,
        'Word2Vec__500' : w2v_param_grid,
        'SentenceTransformer' : sbert_param_grid
    }

    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42) # Si temps de faire des tests d'hypothèse
    # cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # Définition du pipeline de recherche d'hyperparamètres 
    for vectorizer_name, vectorizer in vectorizers.items():
        specific_param_grid = param_grid.get(vectorizer_name, {})

        pipeline = Pipeline([
            ("vectorizer", vectorizer),
            ("classify", "passthrough")
        ])

        grid_search = GridSearchCV(
            pipeline, 
            param_grid=specific_param_grid, 
            verbose=2, 
            cv=cv,
            n_jobs=1, # Éviter la concurrence avec Parallel
            refit='f1_macro', 
            scoring=['accuracy','f1_macro']
        )

        print(f'Running GridSearchCV for {vectorizer_name}...')
        grid_search.fit(X_train, y_train)

        # Stocker les résultats
        results_dic = grid_search.cv_results_
        results_dic['Vectorizer'] = vectorizer_name
        results_dic['Ratio incels'] = int(ratio_incels)
        pd.DataFrame(results_dic).to_csv(f'../results/TEST_results_training_{vectorizer_name}_{ratio_incels}pc_3x_repeated-10folds.csv', index=False)
        results_training.append(results_dic)

**Comparaison statistique des modèles**

Nous allons analyser l'effet de différentes variables sur la performance des modèles de classification :

*Pour les modèles TF-IDF*
1. Le nombre de traits discriminants 
2. Le type de classifieur utilisé 
3. Le ratio de données incels

*Pour les modèles sentence-transformers*
1. Le type de classifieur utilisé
2. Le ratio de données incels

Nous retiendrons ensuite les trois meilleurs modèles pour la suite des analyses.

In [ ]:
modeles_sbert = pd.read_csv('../results/results_training_sbeert_5folds.csv')

In [ ]:
# modeles_tf_idf = results_df[results_df['Vectorizer'] == 'TfidfVectorizer']
# modeles_tf_idf.head()

In [ ]:
# Colonnes des scores individuels F1_macro
score_columns = [col for col in modeles_tf_idf.columns if "split" in col and "test_f1_macro" in col]

# Transformation des données en format long
long_format = pd.melt(
    modeles_tf_idf,
    id_vars=["param_classify", "param_vectorizer__max_features"],
    value_vars=score_columns,
    var_name="Split",
    value_name="test_f1_macro"
)

# Création d'une colonne unique pour chaque configuration (modèle + features)
long_format["Model_Features"] = (
    long_format["param_classify"].astype(str) 
    + "_" 
    + long_format["param_vectorizer__max_features"].astype(str)
)

# Aperçu des données transformées
long_format.head()

ANOVA - Vérification des conditions

1. Indépendance

Cette condition est remplie par la validation croisée où chaque pli est indépendant des autres

*2. Distribution normale des données*

In [ ]:
liste_donnees = [] 
for model in long_format['Model_Features'].unique():
    # Exemple de données pour un groupe
    donnees = long_format[long_format['Model_Features'] == model]['test_f1_macro'].tolist()

    stat, p = shapiro(donnees)

    if p < 0.05:
        print(model, "Les données ne suivent pas une distribution normale.")
        print(f"Statistique de Shapiro-Wilk : {stat}, p-value : {p}")

    liste_donnees.append(donnees)

*3. Homogénéité des variances entre les groupes*

In [ ]:
stat, p = levene(
    *liste_donnees
)
print(f"Statistique de Levene : {stat}, p-value : {p}")

if p > 0.05:
    print("Les variances sont homogènes.")
else:
    print("Les variances ne sont pas homogènes.")


**ANOVA**   
Comme les conditions sont respectées, nous allons utiliser un test ANOVA pour comparer les moyennes de performances entre nos modèles.

*Anova à deux facteurs*: (modifier pour ajouter un troisième facteur plus tard)
1. Classifieur utilisé (Multinomial Naive Bayes, Logistic Regression, Support Vector Machine)
2. Nombre de features (5000, 10 000, 15 000)

In [ ]:
long_format

In [ ]:
modeles_tf_idf[modeles_tf_idf['param_vectorizer__max_features'] == 15000]

In [ ]:
# Importing libraries 
import statsmodels.api as sm 
from statsmodels.formula.api import ols 
  
# Performing two-way ANOVA 
model = ols( 
    'test_f1_macro ~ C(param_classify) + C(param_vectorizer__max_features) +C(param_classify):C(param_vectorizer__max_features)', 
    data=long_format).fit() 


anova_results = sm.stats.anova_lm(model, typ=2) 

# Format the p-values with several decimals
anova_results['reject H0'] = anova_results['PR(>F)'].apply(lambda x: True if x < 0.05 else False)

anova_results

**Test de Tukey (HSD)**

Le résultat au test ANOVA indique que les deux paramètres (classifieur et nombre de traits discriminants) ont un effet significatif (p < 0.05) sur la performance des modèles (f1_macro).  
Nous allons donc maintenant analyser les modèles en paires (comparaisons 1 à 1) pour explorer plus en détail leurs différences. 

In [ ]:
from scipy.stats import tukey_hsd
import re

# Exécuter le test de Tukey
tukey_result = tukey_hsd(*liste_donnees)

# Renommer les indices pour les modèles
model_names = {
    i : long_format['Model_Features'].unique()[i] for i in range(len(long_format['Model_Features'].unique()))
}

In [ ]:
str_results = '\n'.join(str(tukey_result).split('\n')[1:])
str_results = re.sub(r'\s-\s', '-', str_results)
str_results = str_results.replace('Lower CI', 'Lower_CI').replace('Upper CI', 'Upper_CI').replace('(', '').replace(')', '')

with open('../results/tukey_results_tf-idf_models.txt', 'w') as f:
    f.write(str_results)

df = pd.read_csv('../results/tukey_results_tf-idf_models.txt', delim_whitespace=True, header=0)
df[['Model 1', 'Model 2']] = df['Comparison'].str.split('-', n=1, expand=True)

df['Model 1'] = df['Model 1'].astype(int).map(model_names)
df['Model 2'] = df['Model 2'].astype(int).map(model_names)

df['Reject H0'] = df['p-value'].apply(lambda x: True if x < 0.05 else False)

df = df[['Model 1', 'Model 2', 'Statistic', 'p-value', 'Lower_CI', 'Upper_CI', 'Reject H0']]
df.to_csv('../results/tukey_results_tf-idf_models.csv', index=False)

In [ ]:
df = df[df['Model 1'].str.contains('MultinomialNB()')]
df = df[df['Model 2'].str.contains('MultinomialNB()')]

df['n_features_model1'] = df['Model 1'].apply(lambda x: int(float(x.split('_')[1])))
df['n_features_model2'] = df['Model 2'].apply(lambda x: int(float(x.split('_')[1])))

df = df[['n_features_model1', 'n_features_model2', 'p-value', 'Reject H0']]
df.sort_values(by=['n_features_model1', 'n_features_model2'])

La comparaison en paire permet d'observer qu'au-delà le 1000 traits discriminants, les différences de performances entre les modèles ne sont plus statistiquement significatives.   
Nous conserverons donc un maximum de 1000 traits. 

___